---
## Section 1: Setup and architecture

This notebook uses two services for separate jobs:

1. **PageIndex** uses your PageIndex API key to upload/index a PDF and return its hierarchical table of contents (TOC).
2. **Ollama (`qwen2.5:7b`)** runs locally and uses the TOC to select relevant sections, then answers from their text.

No OpenAI key or OpenAI client is used. The PageIndex key is tested with a read-only document-list request before any upload. Ollama is checked separately before the pipeline starts.

Get a PageIndex API key from https://dash.pageindex.ai/api-keys. Install Ollama, then run `ollama pull qwen2.5:7b` before the Ollama check.


In [ ]:
# Install the packages used by this notebook (no OpenAI SDK required)
%pip install pageindex python-dotenv


In [ ]:
# Create a safe .env template once; never overwrite an existing key.
from pathlib import Path

env_path = Path(".env")
if not env_path.exists():
    env_path.write_text("PAGEINDEX_API_KEY=\n", encoding="utf-8")
    print("Created .env. Add your PageIndex key to PAGEINDEX_API_KEY before continuing.")
else:
    print(".env already exists; leaving it unchanged.")


In [ ]:
# Layer 1: Import the PageIndex key from .env and validate that it is present.
import json
import os
import time
from dotenv import load_dotenv

load_dotenv()
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "").strip()
if not PAGEINDEX_API_KEY or PAGEINDEX_API_KEY.lower() in {"your_key_here", "replace_me", "<your_pageindex_api_key>"}:
    raise ValueError("PAGEINDEX_API_KEY is missing. Add your PageIndex key to the .env file, then rerun this cell.")

print("PageIndex key loaded from .env: yes (value hidden)")


In [ ]:
# Layer 2: Test the PageIndex key with a real, read-only API request.
from pageindex import PageIndexClient
from pageindex.errors import PageIndexAPIError

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
try:
    key_test_result = pi_client.list_documents(limit=1)
except PageIndexAPIError as exc:
    status = f" (HTTP {exc.status_code})" if exc.status_code is not None else ""
    raise RuntimeError(
        f"PageIndex key test failed{status}. Check that PAGEINDEX_API_KEY is valid and active."
    ) from exc

print("PageIndex API key test: passed")
print(f"Accessible documents: {key_test_result.get('total', 'unknown')}")

# Layer 3: Configure and test the local Ollama model. No cloud LLM key is used.
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
OLLAMA_MODEL = "qwen2.5:7b"

def ollama_chat(prompt_or_messages, model=OLLAMA_MODEL, json_mode=False):
    """Call a locally running Ollama chat model and return its message text."""
    messages = prompt_or_messages if isinstance(prompt_or_messages, list) else [
        {"role": "user", "content": prompt_or_messages}
    ]
    payload = {"model": model, "messages": messages, "stream": False}
    if json_mode:
        payload["format"] = "json"

    request = Request(
        f"{OLLAMA_BASE_URL}/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=120) as response:
            result = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        if exc.code == 404 and "model" in detail.lower():
            raise RuntimeError(
                f"Ollama model '{model}' is not installed. Run `ollama pull {model}` and retry."
            ) from exc
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(
            f"Cannot reach Ollama at {OLLAMA_BASE_URL}. Start Ollama and retry."
        ) from exc

    message = result.get("message", {}).get("content", "")
    if not message.strip():
        raise RuntimeError("Ollama returned an empty response.")
    return message

ollama_test = ollama_chat("Reply with exactly: Ollama is ready.")
print(f"Ollama model test ({OLLAMA_MODEL}): passed. Response: {ollama_test.strip()}")


---
## Section 2: Upload and index a PDF

**Layer 4 ? PageIndex cloud indexing:**
1. Upload the PDF with the authenticated PageIndex client.
2. PageIndex processes the document and builds its hierarchical tree / table of contents (TOC).
3. Keep the returned `doc_id` for status and tree requests.

The PageIndex API owns TOC creation in this hosted workflow. The local Ollama model is used later for query-time section selection and answer generation; it does not replace PageIndex's cloud indexing service.

**Why no chunking?** PageIndex represents the document using its section hierarchy instead of splitting it into arbitrary fixed-size pieces.


In [ ]:
# ── Upload your PDF ─────────────────────────────────────────────────────────

# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./Test/Placement.pdf"   

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

In [ ]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

---
## Section 3: Inspect the tree structure

The PageIndex response is a nested TOC. Each node has:
- `node_id` ? unique ID used during retrieval
- `title` ? section heading
- `page_index` ? page number in the original PDF
- `text` ? section text or summary, depending on the response
- `nodes` ? child sections

**Layer 5:** The local `qwen2.5:7b` model receives this structure and the question to select relevant node IDs. PageIndex remains the source of the TOC and document sections.


In [51]:
# Fetch the PageIndex tree and page-level source text. The OCR/page text is
# the evidence source; tree summaries alone are not enough to answer questions.
tree_result = pi_client.get_tree(doc_id, node_summary=True, include_text=True)
if tree_result.get("status") not in {None, "completed"}:
    raise RuntimeError(f"PageIndex tree is not ready: {tree_result.get('status')}")
pageindex_tree = tree_result.get("result") or []
if not pageindex_tree:
    raise RuntimeError("PageIndex returned an empty tree. Check that document processing completed.")

ocr_result = pi_client.get_ocr(doc_id, format="page")
ocr_pages = ocr_result.get("result") or []
page_text_by_index = {
    int(page["page_index"]): page.get("markdown", "")
    for page in ocr_pages
    if page.get("page_index") is not None and page.get("markdown", "").strip()
}

print(f"Top-level sections: {len(pageindex_tree)}")
print(f"Pages with extracted text: {len(page_text_by_index)}")
if not page_text_by_index:
    print("Warning: PageIndex returned no page text. The answer step can only use text embedded in tree nodes.")


Top-level sections: 12
Pages with extracted text: 9


In [52]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] PLACEMENT AND INTERNSHIP POLICY 2026-27  (p.?)
[0001] 2. Eligibility for Placement through Placement Cell, CDC.  (p.?)
[0002] 3. Rules for Applying for Placement Process.  (p.?)
[0003] 9. Semester-long / 6 months Internship & Performance-based PPO:  (p.?)
[0004] 10. Semester-long / 6 months Internship (8B) Registration Rules:  (p.?)
[0005] Category I: Internship with Pre-Placement Offer (PPO)/Performance-Based Conversion  (p.?)
[0006] Category II: On-Campus Placed Students willing to accept Independent Off-Campus Internship  (p.?)
[0007] Category III: Students Without Any Confirmed Full-Time Employment Offer  (p.?)
[0008] 13. Collision of PPO Offers and Off-campus Company Offers  (p.?)
[0009] c) Non-Compliance  (p.?)
[0010] 17. Fine for Not Joining the Company After Acceptance of Offer  (p.?)
[0011] 19. General Advice to Students/Placement Conveners.  (p.?)


In [ ]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

---
## Section 4: Local Ollama tree search

PageIndex provides a structured TOC instead of a flat set of vector chunks. The notebook sends the question and a compact TOC to **local Ollama (`qwen2.5:7b`)**:

```text
question + PageIndex TOC ? local Ollama ? relevant node IDs
```

Ollama returns a concise selection summary and the node IDs. The next layer uses those IDs to retrieve section text and page references from the PageIndex tree.


In [53]:
# Layer 5: Ask Ollama to choose relevant TOC nodes. A deterministic fallback
# keeps retrieval moving if a small local model returns an empty node list.

def llm_tree_search(query: str, tree: list, model: str = OLLAMA_MODEL) -> dict:
    """Select relevant nodes from the PageIndex TOC, with a safe local fallback."""
    def compress(nodes):
        compact = []
        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node.get("title", ""),
                "pages": f"{node.get('start_index', node.get('page_index', '?'))}"
                + (f"-{node['end_index']}" if node.get("end_index") is not None
                   and node.get("end_index") != node.get("start_index", node.get("page_index")) else ""),
                "summary": node.get("summary") or node.get("text", "")[:500],
            }
            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])
            compact.append(entry)
        return compact

    prompt = f"""Select the best matching document sections for this question.
Do not answer. Unless the document tree is empty, always return at least one node ID.
For a broad question, choose the most relevant top-level section(s). Choose up to 5.
Return JSON with "selection_summary" and "node_list" (existing node ID strings).

Question: {query}

PageIndex table of contents:
{json.dumps(compress(tree), ensure_ascii=False, indent=2)}"""

    try:
        result = json.loads(ollama_chat(prompt, model=model, json_mode=True))
    except (json.JSONDecodeError, TypeError):
        result = {}
    if not isinstance(result, dict):
        result = {}

    all_nodes = []
    def collect(nodes):
        for node in nodes:
            all_nodes.append(node)
            collect(node.get("nodes", []))
    collect(tree)
    valid_ids = {node["node_id"] for node in all_nodes}

    selected = []
    for node_id in result.get("node_list", []):
        if isinstance(node_id, str) and node_id in valid_ids and node_id not in selected:
            selected.append(node_id)

    if not selected and all_nodes:
        stop_words = {"what", "when", "where", "which", "who", "whom", "why", "how",
                      "the", "and", "for", "from", "with", "that", "this", "are", "was",
                      "were", "can", "could", "would", "should", "does", "did", "has", "have",
                      "been", "being", "already", "another", "about", "document", "please"}
        terms = {word for word in query.casefold().replace("?", " ").replace("/", " ").split()
                 if len(word) > 2 and word not in stop_words}

        def score(node):
            words = set((node.get("title", "") + " " + node.get("summary", "") + " " +
                         node.get("text", "")[:1000]).casefold().replace("-", " ").split())
            return len(terms & words)

        ranked = sorted(all_nodes, key=score, reverse=True)
        relevant = [node for node in ranked if score(node) > 0][:5]
        if not relevant:
            leaves = [node for node in all_nodes if not node.get("nodes")]
            relevant = leaves[:3] if leaves else tree[:3]
        selected = [node["node_id"] for node in relevant]
        result["selection_summary"] = "Used fallback section selection because Ollama returned no usable node IDs."

    # Add lexical matches so a plausible but off-topic Ollama choice cannot
    # suppress a directly matching policy clause (for example, offer revocation).
    stop_words = {"what", "when", "where", "which", "who", "whom", "why", "how",
                  "the", "and", "for", "from", "with", "that", "this", "are", "was",
                  "were", "can", "could", "would", "should", "does", "did", "has", "have",
                  "been", "being", "already", "another", "about", "document", "please",
                  "student", "students", "company", "companies", "offer", "placement",
                  "internship", "campus", "on", "off", "institute", "mediated"}
    terms = {word for word in query.casefold().replace("?", " ").replace("/", " ").split()
             if len(word) > 2 and word not in stop_words}
    def lexical_score(node):
        text = " ".join((node.get("title", ""), node.get("summary", ""),
                         node.get("text", "")))
        words = set(text.casefold().replace("-", " ").replace("/", " ").split())
        score = sum(term in words for term in terms)
        if "revok" in " ".join(terms) and any(word.startswith("revok") or word.startswith("revoc") for word in words):
            score += 3
        if {"before", "joining"}.issubset(terms) and "join" in " ".join(words):
            score += 2
        return score

    lexical_matches = sorted(all_nodes, key=lexical_score, reverse=True)
    lexical_matches = [node for node in lexical_matches if lexical_score(node) > 0][:3]
    for node in lexical_matches:
        if node["node_id"] not in selected:
            selected.append(node["node_id"])
    result["node_list"] = selected[:6]
    if lexical_matches:
        result["selection_summary"] = (
            result.get("selection_summary", "") +
            " Lexical policy-term matches were added to improve recall."
        ).strip()
    result.setdefault("selection_summary", "Selected sections from the PageIndex TOC.")
    return result


In [54]:
# Layer 6: Try TOC-based retrieval on a sample question.
query = "if a person is already placed for a 6 month internship, can he/she apply for another company?"

print(f"Question: {query}\n")
result = llm_tree_search(query, pageindex_tree)
print("Selection summary:", result.get("selection_summary", "Not provided"))
print("Selected node IDs:", result.get("node_list", []))


Question: if a person is already placed for a 6 month internship, can he/she apply for another company?

Selection summary: The question is broad and general, so the most relevant top-level sections are the ones that provide an overview of the placement and internship policies. The top-level sections that are most relevant are sections that discuss general rules and categories, which are found in sections 3 and 10. Lexical policy-term matches were added to improve recall.
Selected node IDs: ['0002', '0004', '0008', '0003']


---
## Section 5: End-to-end RAG pipeline

**Three query-time steps:**
1. **Tree search** ? local Ollama proposes relevant PageIndex sections; lexical policy-term matching adds likely missed clauses.
2. **Retrieve** ? fetch extracted PDF pages from the selected section ranges, then keep the pages whose text best matches the question.
3. **Answer with evidence checks** ? Ollama answers only from those pages; Python verifies its evidence quote and cites the source pages.

The hierarchical TOC helps narrow the search, while page-level PDF text is the actual answer evidence. If the model's evidence cannot be verified, the notebook says so instead of presenting an unsupported answer.


In [55]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [56]:
# Layer 7: Retrieve question-matched PDF pages, then verify Ollama's evidence.
def generate_answer(query: str, nodes: list, model: str = OLLAMA_MODEL,
                    page_texts: dict = None) -> str:
    """Generate an answer from the best-matching PDF pages and verify its quote."""
    import re
    page_texts = page_texts or {}
    if not nodes:
        return "No relevant sections were selected. Try a more specific question."

    stop_words = {"what", "when", "where", "which", "who", "whom", "why", "how",
                  "the", "and", "for", "from", "with", "that", "this", "are", "was",
                  "were", "can", "could", "would", "should", "does", "did", "has", "have",
                  "been", "being", "already", "about", "document", "please", "student",
                  "students", "they", "their", "them", "he", "she", "company", "companies", "offer", "placement", "internship",
                  "campus", "institute", "mediated", "apply", "another", "six", "month"}
    raw_terms = {word for word in re.findall(r"[a-z0-9]+", query.casefold()) if len(word) > 2}
    terms = {word for word in raw_terms if word not in stop_words}
    if raw_terms & {"another", "second"}:
        terms.add("subsequent")
    if raw_terms & {"apply", "participate"}:
        terms.add("eligible")
    if "secured" in raw_terms or "securing" in raw_terms:
        terms.add("secures")
    if any(word.startswith("revok") or word.startswith("withdraw") for word in terms):
        terms.update({"revoked", "revocation"})
    if "joining" in terms or "join" in terms:
        terms.add("joining")

    node_by_id = {node["node_id"]: node for node in nodes}
    page_owners = {}
    for node in nodes:
        start = node.get("start_index", node.get("page_index"))
        end = node.get("end_index", start)
        if isinstance(start, int):
            last = end if isinstance(end, int) else start
            for page in page_texts:
                if start <= page <= last:
                    page_owners.setdefault(page, []).append(node)

    def relevance(text):
        words = set(re.findall(r"[a-z0-9]+", text.casefold()))
        score = sum(term in words for term in terms)
        if "subsequent" in terms and "subsequent" in words:
            score += 2
        if "revoked" in terms and ("revoked" in words or "revocation" in words):
            score += 2
        return score

    selected_pages = []
    if page_owners:
        ranked_pages = sorted(page_owners, key=lambda page: relevance(page_texts[page]), reverse=True)
        matches = [page for page in ranked_pages if relevance(page_texts[page]) > 0]
        selected_pages = (matches or ranked_pages)[:2]

    sources = {}
    context_parts = []
    for page in selected_pages:
        candidates = page_owners[page]
        node = max(candidates, key=lambda item: relevance(
            item.get("title", "") + " " + item.get("summary", "")
        ))
        node_id = node["node_id"]
        title = node.get("title", "Untitled section")
        text = page_texts[page]
        if node_id not in sources:
            sources[node_id] = {"title": title, "pages": [], "page_texts": {}, "text": ""}
        sources[node_id]["pages"].append(page)
        sources[node_id]["page_texts"][page] = text
        sources[node_id]["text"] += f"\n\n[PDF page {page}]\n{text}"
        context_parts.append(f"[NODE_ID: {node_id} | PDF PAGE: {page}]\n{text}")

    if not context_parts:
        for node in nodes:
            text = (node.get("text") or "").strip()
            if not text:
                continue
            node_id = node["node_id"]
            sources[node_id] = {
                "title": node.get("title", "Untitled section"),
                "pages": [node.get("start_index", node.get("page_index", "unknown"))],
                "page_texts": {},
                "text": text,
            }
            context_parts.append(f"[NODE_ID: {node_id}]\n{text[:5000]}")

    if not context_parts:
        return "PageIndex selected sections, but no source text was available. Re-run the tree and page-text cell."

    prompt = f"""Answer this question only from the policy text. Give 1-2 sentences explaining
the rule; do not answer with only yes or no. Return JSON with "answer",
"insufficient_evidence" (boolean), and "evidence" (an array containing one
object with "node_id" and a short exact quote from that node). Evidence MUST be
an array, not a string. Do not confuse separate rules. If a specific clause
directly answers the question, prioritize it over general or unrelated
exceptions. Answer only the question asked; do not add an unrelated off-campus
rule or exception. Do not add facts not stated in the policy.

Question: {query}

Policy text:\n{chr(10).join(context_parts)}"""
    raw = ollama_chat(prompt, model=model, json_mode=True)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {}

    def normalize(text):
        return " ".join(re.findall(r"[a-z0-9]+", text.casefold()))

    def verify_result(result):
        if not isinstance(result, dict) or result.get("insufficient_evidence") is True:
            return None
        answer = result.get("answer")
        evidence = result.get("evidence")
        if isinstance(evidence, str):
            evidence = [{"quote": evidence}]
        elif isinstance(evidence, list):
            evidence = [{"quote": item} if isinstance(item, str) else item for item in evidence]
        else:
            evidence = []
        if not isinstance(answer, str) or not answer.strip() or not evidence:
            return None

        cited_ids = []
        citation_pages = {}
        for item in evidence:
            if not isinstance(item, dict):
                return None
            quote = item.get("quote")
            node_id = item.get("node_id")
            if not isinstance(quote, str) or not quote.strip():
                return None
            if node_id not in sources:
                matches = [source_id for source_id, source in sources.items()
                           if normalize(quote) in normalize(source["text"])]
                if not matches:
                    return None
                node_id = matches[0]
            if normalize(quote) not in normalize(sources[node_id]["text"]):
                return None
            matching_pages = [
                page for page, page_text in sources[node_id]["page_texts"].items()
                if normalize(quote) in normalize(page_text)
            ]
            citation_pages.setdefault(node_id, set()).update(
                matching_pages or sources[node_id]["pages"]
            )
            if node_id not in cited_ids:
                cited_ids.append(node_id)

        citations = "; ".join(
            f"{sources[node_id]['title']} (PDF page(s) {', '.join(map(str, sorted(citation_pages[node_id], key=str)))})"
            for node_id in cited_ids
        )
        return f"{answer.strip()}\n\nSources: {citations}"

    verified = verify_result(result)
    if verified:
        return verified

    # Small local models occasionally return an empty or malformed JSON object.
    # Retry once with a compact prompt before surfacing an explicit fallback.
    retry_prompt = f"""Use only this policy excerpt to answer: {query}
Return JSON exactly as {{"answer":"...","insufficient_evidence":false,"evidence":[{{"node_id":"...","quote":"exact short quote"}}]}}.
The evidence value must be an array. Copy a short quote exactly from this text:
{chr(10).join(context_parts)[:6500]}"""
    retry_raw = ollama_chat(retry_prompt, model=model, json_mode=True)
    try:
        verified = verify_result(json.loads(retry_raw))
    except json.JSONDecodeError:
        verified = None
    if verified:
        return verified
    return "I couldn't verify a supported answer from the selected policy text. Review the relevant PDF pages shown above."


In [57]:
# Layer 8: Retrieve selected sections' PageIndex text and answer from PDF pages.

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """Run TOC search, page-text retrieval, and quote-verified answer generation."""
    if verbose:
        print(f"{'=' * 55}\nQuestion: {query}\n{'=' * 55}")

    search_result = llm_tree_search(query, tree)
    nodes = find_nodes_by_ids(tree, search_result["node_list"])

    if verbose:
        print("\nSelection summary:", search_result.get("selection_summary", "Not provided"))
        print("Selected sections:", [node.get("title", "Untitled section") for node in nodes])

    answer = generate_answer(query, nodes, page_texts=page_text_by_index)
    if verbose:
        print("\nAnswer:\n", answer)
    return answer


In [62]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query= "lets say if a person gets selected for google his dream company offcampus but he already holda an offer from cisco for fte, then what choices does he have and what actions will be taken on him by the college if he doesnot joins cisco on time?",
    tree=pageindex_tree
)

Question: lets say if a person gets selected for google his dream company offcampus but he already holda an offer from cisco for fte, then what choices does he have and what actions will be taken on him by the college if he doesnot joins cisco on time?

Selection summary: The question relates to a student's options when they have both an on-campus and an off-campus offer, and the consequences if they do not join the on-campus offer on time. The most relevant sections are those that deal with the handling of multiple offers and the rules for joining companies after accepting an offer. Lexical policy-term matches were added to improve recall.
Selected sections: ['3. Rules for Applying for Placement Process.', 'Category I: Internship with Pre-Placement Offer (PPO)/Performance-Based Conversion', 'Category II: On-Campus Placed Students willing to accept Independent Off-Campus Internship', '13. Collision of PPO Offers and Off-campus Company Offers', 'c) Non-Compliance', '17. Fine for Not Joi

In [63]:
# Test the pipeline with questions from the uploaded placement policy.
test_queries = [
    "Can a student who has secured an institute-mediated on-campus internship apply for another on-campus internship?",
    "What does the One Student, One Job and One Student, One Internship rule say?",
    "If an on-campus internship offer is revoked before joining, may the student participate in future placement opportunities?",
    "For a six-month internship with a performance-based PPO, when may the student continue FTE placement and what must they do after receiving an FTE offer?",
    "Can a student with a confirmed on-campus full-time offer take an independent six-month off-campus internship, and what undertaking is required?",
    "How must students register for a company placement process, and what happens if they stop participating after registering?",
]

for question in test_queries:
    print(f"\nQuestion: {question}")
    print(vectorless_rag(question, pageindex_tree, verbose=False))
    print("-" * 70)



Question: Can a student who has secured an institute-mediated on-campus internship apply for another on-campus internship?
A student who has secured an Institute-mediated on-campus internship is not eligible to participate in any subsequent Institute-mediated on-campus internship processes.

Sources: 3. Rules for Applying for Placement Process. (PDF page(s) 3)
----------------------------------------------------------------------

Question: What does the One Student, One Job and One Student, One Internship rule say?
The 'One Student, One Job' and 'One Student, One Internship' policy restricts students from participating in subsequent institute-mediated campus placement processes once they have secured a job or internship through any such process.

Sources: 3. Rules for Applying for Placement Process. (PDF page(s) 3)
----------------------------------------------------------------------

Question: If an on-campus internship offer is revoked before joining, may the student participate i

---
## 🎓 Section 6: Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.


In [ ]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.


In [64]:
PLACEMENT_EXPERT_RULES = """
Route placement-policy questions to these topics:
- One student / one job or one internship, later company eligibility -> Section 6(a)
- Company result timing, multiple selections, choosing one offer -> Section 6(b)
- Offer revoked before joining -> Section 6(c)
- Dream Company eligibility -> Section 7
- Six-month internship with performance-based PPO and later FTE placement -> Section 9
- 8B registration categories and undertakings -> Section 10 and its categories
- Offer terms and acceptance -> Section 11
- PPO / off-campus offer collision -> Section 13
- Registration, attendance, withdrawal, or skipping processes -> Sections 3-5

Use only the uploaded policy. Do not treat an off-campus internship exception as an
exception to the on-campus one-internship rule unless the policy explicitly says so.
"""
print("Placement-policy routing rules ready.")


Placement-policy routing rules ready.


In [67]:
# Topic routing is specific to the uploaded NIT Raipur placement policy.
PLACEMENT_EXPERT_RULES = """
- Later on-campus placement/internship after selection: Section 6(a), One Student, One Job / One Student, One Internship.
- Multiple selections before final selection results: Section 6(b), choice and 3-day notification.
- Placement or internship offer revoked before joining: Section 6(c), future eligibility.
- Six-month internship with performance-based PPO and FTE placement date: Section 9.
- Six-month internship / 8B registration, PPO conversion, on-campus placed Category II, no confirmed FTE Category III: Section 10 and relevant category.
- Offer acceptance and conditions: Section 11.
- PPO and off-campus company offer conflict: Section 13.
- Registration, withdrawal, attendance, and process participation: Sections 3-5.

Always distinguish on-campus from off-campus internships and internship offers from
full-time employment offers. Quote the policy and say when it does not resolve a question.
"""


In [68]:
# Expert-guided TOC search also runs entirely through local Ollama.

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = OLLAMA_MODEL,
) -> dict:
    """Use the rules as additional guidance when selecting TOC node IDs."""
    def compress(nodes):
        compressed = []
        for node in nodes:
            entry = {
                "node_id": node["node_id"],
                "title": node["title"],
                "page": node.get("page_index", "?"),
                "summary": node.get("text", "")[:150],
            }
            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])
            compressed.append(entry)
        return compressed

    prompt = f"""Select the document sections most likely to answer the question.
Use the expert rules as guidance. Return a concise selection summary, not
step-by-step reasoning. Choose only node IDs present in the TOC. Return a JSON
object with exactly these keys: "selection_summary" (string) and "node_list"
(array of node ID strings).

Question: {query}

Document TOC:
{json.dumps(compress(tree), indent=2)}

Expert rules:
{expert_rules}"""
    result = json.loads(ollama_chat(prompt, model=model, json_mode=True))
    if not isinstance(result, dict) or not isinstance(result.get("node_list"), list):
        raise ValueError("Ollama response must be a JSON object containing a node_list array.")
    return result


In [69]:
# Compare basic and placement-policy-guided retrieval.
query = "Can a student already selected for an on-campus internship apply to another company?"
print(f"Question: {query}\n")

print("Without placement-specific routing:")
basic = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print("\nWith placement-specific routing:")
guided = llm_tree_search_with_expert(query, pageindex_tree, PLACEMENT_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Selection summary:", guided.get("selection_summary", "Not provided"))


Question: Can a student already selected for an on-campus internship apply to another company?

Without placement-specific routing:
Nodes: ['0006', '0002']

With placement-specific routing:
Nodes: ['0006', '0008']
Selection summary: The question pertains to whether a student can apply to another company after being selected for an on-campus internship. This is most relevant to the rules outlined in Section 6(c) of the policy.


In [70]:
# Expert-guided question about a real placement-policy rule.
def expert_rag(query: str, tree: list, rules: str) -> str:
    """Combine expert and general retrieval before checking source pages."""
    expert_result = llm_tree_search_with_expert(query, tree, rules)
    general_result = llm_tree_search(query, tree)
    node_ids = list(dict.fromkeys(
        expert_result.get("node_list", []) + general_result.get("node_list", [])
    ))
    nodes = find_nodes_by_ids(tree, node_ids)
    return generate_answer(query, nodes, page_texts=page_text_by_index)

answer = expert_rag(
    query="Can a student who already secured an institute-mediated on-campus internship participate in a later on-campus internship process?",
    tree=pageindex_tree,
    rules=PLACEMENT_EXPERT_RULES,
)
print(answer)


A student who already secured an institute-mediated on-campus internship cannot participate in a later on-campus internship process.

Sources: 3. Rules for Applying for Placement Process. (PDF page(s) 3)


---
## Section 7: Multi-turn chat with local Ollama

Each turn uses the current question to select relevant sections from the PageIndex TOC. Those sections and the conversation history are then sent to your local `qwen2.5:7b` model. PageIndex is used for document indexing and TOC access; Ollama handles the LLM work.


In [ ]:
# Ask a placement-policy question with local Ollama.
question = "What does the policy say if an on-campus offer is revoked before the joining date?"
answer = vectorless_rag(question, pageindex_tree)


In [ ]:
# Multi-turn chat about the uploaded placement policy.
conversation_history = []

def chat_with_doc(user_message: str, tree: list) -> str:
    """Answer follow-ups using PageIndex-selected policy pages and local Ollama."""
    search_result = llm_tree_search(user_message, tree)
    nodes = find_nodes_by_ids(tree, search_result["node_list"])
    context_parts = []
    for node in nodes:
        start_page = node.get("start_index", node.get("page_index"))
        end_page = node.get("end_index", start_page)
        if isinstance(start_page, int):
            last_page = end_page if isinstance(end_page, int) else start_page
            for page in sorted(page for page in page_text_by_index if start_page <= page <= last_page)[:8]:
                page_text = page_text_by_index[page]
                if page_text.strip():
                    context_parts.append(
                        f"[Section: {node.get('title', 'Untitled section')} | PDF page {page}]\n{page_text}"
                    )
        elif node.get("text", "").strip():
            context_parts.append(
                f"[Section: {node.get('title', 'Untitled section')}]\n{node['text']}"
            )

    if not context_parts:
        return "No policy text was retrieved for the selected sections. Re-run the PageIndex tree and page-text cell."

    messages = [
        {
            "role": "system",
            "content": (
                "Answer only from the supplied NIT Raipur placement-policy excerpts. "
                "Distinguish on-campus internships from off-campus internships and full-time offers. "
                "Cite the relevant section and PDF page; say if the policy is unclear.\n\n"
                + "\n\n---\n\n".join(context_parts)
            ),
        },
        *conversation_history[-6:],
        {"role": "user", "content": user_message},
    ]
    assistant_reply = ollama_chat(messages)
    conversation_history.extend([
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_reply},
    ])
    return assistant_reply

questions = [
    "Can I apply for another on-campus internship after getting one?",
    "What if the internship offer is revoked before joining?",
    "Does the rule for an independent off-campus internship change that answer?",
]
for question in questions:
    print(f"\nStudent: {question}")
    print(f"Ollama: {chat_with_doc(question, pageindex_tree)}")
    print("-" * 55)


---
## Section 8: Vector RAG vs PageIndex

In this notebook, PageIndex provides a hierarchical document tree and local Ollama handles query-time reasoning. This avoids a vector database while preserving section names and page references.

| Aspect | Traditional Vector RAG | This notebook |
|--------|------------------------|---------------|
| Document preparation | Split into chunks and embed | PageIndex creates a hierarchical tree |
| Query processing | Similarity search over vectors | Local `qwen2.5:7b` selects TOC nodes |
| Answer generation | LLM over retrieved chunks | Local `qwen2.5:7b` over selected section text |
| Traceability | Depends on chunk metadata | Section titles and page references |
| Vector database | Usually required | Not used |


In [ ]:
# ── Quick comparison demo ────────────────────────────────────────────────────
# Show how the same query retrieves differently

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context
""")

print("=" * 55)
print("PAGEINDEX approach (actual):")
print("=" * 55)
print("""
result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number
""")

---
## Section 9: Cleanup

Delete the uploaded document from the PageIndex cloud when you no longer need its index. The delete call below stays commented out until you choose to run it.


In [ ]:
# ── Delete document from cloud ───────────────────────────────────────────────
# WARNING: This permanently deletes the tree index.
# Comment this out if you want to reuse the doc_id later.

# pi_client.delete_document(doc_id)
# print(f"🗑️ Deleted document: {doc_id}")
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")

---
## Summary

This notebook uses a single API credential and a local LLM:

1. Load `PAGEINDEX_API_KEY` from `.env` without displaying it.
2. Test the key with a read-only PageIndex document-list request.
3. Verify local Ollama is reachable and `qwen2.5:7b` can respond.
4. Upload a PDF to PageIndex and retrieve its hierarchical TOC.
5. Use Ollama to select tree nodes and generate answers grounded in their text.

No OpenAI key or PageIndex chat API is needed for the query-time RAG and chat examples. Ollama must be installed and running locally; PageIndex cloud indexing requires the PageIndex API key.

### Resources
- PageIndex API keys: https://dash.pageindex.ai/api-keys
- Ollama: https://ollama.com
- PageIndex documentation: https://docs.pageindex.ai
